In [42]:
# REFACTORIZACION DE CODIGO
# ----------------------------------------------------------------
# Carga de datos


def load_data():

    import pandas as pd

    INPUT_FILE = "../data/winequality-red.csv"

    df = pd.read_csv(INPUT_FILE)

    y = df["quality"]
    x = df.copy()
    x.pop("quality")

    return x, y

In [43]:
import os
import pickle

import numpy as np
import pandas as pd
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split

INPUT_FILE = "../data/winequality-red.csv"
OUTPUT_FOLDER = "../submission"
ESTIMATOR_FILE = f"{OUTPUT_FOLDER}/estimator.pkl"


def load_data():
    df = pd.read_csv(INPUT_FILE)
    y = df["quality"]
    x = df.drop(columns="quality")
    return x, y


def make_train_test_split(x, y):
    return train_test_split(x, y, test_size=0.25, random_state=123456)


def eval_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mse, mae, r2


def report(estimator, mse, mae, r2):
    print(estimator, ":", sep="")
    print(f"  MSE: {mse}")
    print(f"  MAE: {mae}")
    print(f"  R2: {r2}")


def load_best_estimator():
    if not os.path.exists(ESTIMATOR_FILE):
        return None
    with open(ESTIMATOR_FILE, "rb") as file:
        return pickle.load(file)


def save_best_estimator(estimator):
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    with open(ESTIMATOR_FILE, "wb") as file:
        pickle.dump(estimator, file)


def train_estimator(alphas, l1_ratios, n_splits=5, verbose=1):
    x, y = load_data()
    x_train, x_test, y_train, y_test = make_train_test_split(x, y)

    search = GridSearchCV(
        estimator=ElasticNet(random_state=12345),
        param_grid={"alpha": alphas, "l1_ratio": l1_ratios},
        cv=n_splits,
        refit=True,
        verbose=0,
        return_train_score=False,
    )
    search.fit(x_train, y_train)
    estimator = search.best_estimator_

    mse, mae, r2 = eval_metrics(y_test, estimator.predict(x_test))
    if verbose > 0:
        report(estimator, mse, mae, r2)

    best_estimator = load_best_estimator()
    if best_estimator is None or estimator.score(x_test, y_test) > best_estimator.score(x_test, y_test):
        best_estimator = estimator

    save_best_estimator(best_estimator)


def check_estimator():
    x, y = load_data()
    _, x_test, _, y_test = make_train_test_split(x, y)
    estimator = load_best_estimator()
    if estimator is None:
        raise RuntimeError("No hay modelo guardado; ejecuta train_estimator primero.")
    mse, mae, r2 = eval_metrics(y_test, estimator.predict(x_test))
    report(estimator, mse, mae, r2)


if __name__ == "__main__":
    train_estimator(
        alphas=np.linspace(0.0001, 0.5, 10),
        l1_ratios=np.linspace(0.0001, 0.5, 10),
        n_splits=5,
        verbose=1,
    )
    check_estimator()  # debe coincidir con el mejor modelo encontrado

ElasticNet(alpha=0.0001, l1_ratio=0.16673333333333332, random_state=12345):
  MSE: 0.4555342217989927
  MAE: 0.5292810437627216
  R2: 0.34641239384627454
ElasticNet(alpha=0.0001, random_state=12345):
  MSE: 0.4391279097984003
  MAE: 0.5194211281128072
  R2: 0.3699517058740983


In [44]:
# Carga del modelo


def load_best_estimator():

    import os
    import pickle

    OUTPUT_FOLDER = "../submission"
    ESTIMATOR_FILE = f"{OUTPUT_FOLDER}/estimator.pkl"

    if not os.path.exists(ESTIMATOR_FILE):
        return None
    with open(ESTIMATOR_FILE, "rb") as file:
        estimator = pickle.load(file)

    return estimator

In [45]:
train_estimator(0.5, 0.5)    


TypeError: Parameter grid for parameter 'alpha' needs to be a list or a numpy array, but got 0.5 (of type float) instead. Single values need to be wrapped in a list with one element.

In [ ]:
# Chequeo
def check_estimator():

    x, y = load_data()
    _, x_test, _, y_test = make_train_test_split(x, y)
    estimator = load_best_estimator()
    mse, mae, r2 = eval_metrics(y_test, y_pred=estimator.predict(x_test))
    report(estimator, mse, mae, r2)


# Debe coincidir con el mejor modelo encontrado en la celdas anteriores
check_estimator()

Modelo: ElasticNet(alpha=0.5, random_state=12345)
  MSE: 0.5978
  MAE: 0.6183
  R2:  0.0979


In [ ]:
# Búsqueda iterativa de hiperparámetros


def make_hyperparameters_search(alphas, l1_ratios):

    for alpha in alphas:
        for l1_ratio in l1_ratios:
            train_estimator(alpha=alpha, l1_ratio=l1_ratio, verbose=0)

In [ ]:
import numpy as np

alphas = np.linspace(0.0001, 0.5, 10)
l1_ratios = np.linspace(0.0001, 0.5, 10)
make_hyperparameters_search(alphas, l1_ratios)
check_estimator()         

Modelo: ElasticNet(alpha=0.0001, random_state=12345)
  MSE: 0.4708
  MAE: 0.5322
  R2:  0.2895


In [ ]:
# Búsqueda usando GridSearchCV


def train_estimator(alphas, l1_ratios, n_splits=5, verbose=1):

    from sklearn.linear_model import ElasticNet
    from sklearn.model_selection import GridSearchCV

    x, y = load_data()
    x_train, x_test, y_train, y_test = make_train_test_split(x, y)

    # -------------------------------------------------------------------------
    # Búsqueda de parámetros con validación cruzada
    #
    estimator = GridSearchCV(
        estimator=ElasticNet(
            random_state=12345,
        ),
        param_grid={
            "alpha": alphas,
            "l1_ratio": l1_ratios,
        },
        cv=n_splits,
        refit=True,
        verbose=0,
        return_train_score=False,
    )
    # -------------------------------------------------------------------------

    estimator.fit(x_train, y_train)

    estimator = estimator.best_estimator_

    mse, mae, r2 = eval_metrics(y_test, y_pred=estimator.predict(x_test))
    if verbose > 0:
        report(estimator, mse, mae, r2)

    best_estimator = load_best_estimator()
    if best_estimator is None or estimator.score(x_test, y_test) > best_estimator.score(
        x_test, y_test
    ):
        best_estimator = estimator


    save_best_estimator(best_estimator)

In [ ]:
import numpy as np

train_estimator(
    alphas=np.linspace(0.0001, 0.5, 10),
    l1_ratios=np.linspace(0.0001, 0.5, 10),
    n_splits=5,
    verbose=1,
)

Modelo: ElasticNet(alpha=0.0001, random_state=12345)
  MSE: 0.4708
  MAE: 0.5322
  R2:  0.2895
